# Viewing MUR layers on the MAAP map

The pipeline's containers write STAC metadata alongside their outputs, and MAAP
ingests it automatically. So the layers below come out of a **catalogue**, not
out of a bucket listing: no credentials, no boto3, no knowing where a DPS job
happened to put its files.

* catalogue  <https://dps-stac.maap-project.org>
* browser    <https://dps-stac-browser.maap-project.org>
* tiles      <https://titiler-dps-stac.maap-project.org>

This runs anywhere with network access, not only in the ADE. It falls back to
plain `ipyleaflet` if `stac_ipyleaflet` is not installed.


In [ ]:
import requests

STAC    = "https://dps-stac.maap-project.org"
TITILER = "https://titiler-dps-stac.maap-project.org"

# MAAP names every collection <username>__<algorithm>__<version>, so the
# version is part of the collection -- a new algorithm version is a new
# collection, not an addition to this one.
USERNAME = "jleach_jpl"
VERSION  = "2.0.13"

BROWSE   = f"{USERNAME}__mur-cog__{VERSION}"     # the GeoTIFF layers
GRANULES = f"{USERNAME}__mur-mrva__{VERSION}"    # the NetCDF products

# Which day to draw. Browse items are <day>-<resolution>; mode is deliberately
# absent, so a REA reprocess replaces the picture for that day rather than
# adding a second layer for the same date.
DAY, RESOLUTION = "20261006", "1km"
item_id = f"mur-cog-{DAY}-{RESOLUTION}"

item = requests.get(f"{STAC}/collections/{BROWSE}/items/{item_id}",
                    timeout=60).json()
if "assets" not in item:
    raise SystemExit(
        f"{item_id} is not in {BROWSE}.\n"
        f"{item.get('detail', item)}\n"
        "Ingestion happens after stage-out, on MAAP's schedule, so a job that "
        "just finished may not be indexed yet. Check the collection list:\n"
        f"  {STAC}/collections")

print(item["id"], "-", item["properties"]["datetime"])
for key, asset in sorted(item["assets"].items()):
    print(f"  {key:10} {asset['href'].rsplit('/', 1)[-1]}")


Each raster asset carries a `mur:render` block with an explicit `rescale`.
That matters more than the colormap: titiler otherwise stretches every tile to
its own data range, which produces a plausible-looking SST field whose colours
mean something different each day.


In [ ]:
def tile_url(asset_key, render):
    """XYZ template for one asset of this item, via its TileJSON.

    Addressed through the catalogue rather than by object URL, so the layer
    keeps working if the underlying object moves -- and so anyone who can see
    the collection can draw it without read access to the bucket.
    """
    url = (f"{TITILER}/collections/{BROWSE}/items/{item_id}"
           f"/WebMercatorQuad/tilejson.json")
    r = requests.get(url, params={"assets": asset_key, **render}, timeout=60)
    r.raise_for_status()
    return r.json()["tiles"][0]


tiles = {}
for key, asset in sorted(item["assets"].items()):
    render = asset.get("mur:render")
    if not render:
        continue
    try:
        tiles[key] = tile_url(key, render)
        print(f"  ok    {key}")
    except Exception as exc:
        # A field can be absent without the run having failed: the COG step
        # treats a field it cannot convert as a missing picture, not an error.
        print(f"  SKIP  {key}: {exc}")


In [ ]:
try:
    from stac_ipyleaflet import StacIpyleaflet
    m = StacIpyleaflet(center=(20, -40), zoom=3)
    add = lambda url, name: m.add_tile_layer(url=url, name=name)
except ImportError:
    from ipyleaflet import Map, TileLayer
    m = Map(center=(20, -40), zoom=3)
    add = lambda url, name: m.add_layer(TileLayer(url=url, name=name))

for key, url in tiles.items():
    add(url, key)
m


## If a layer is blank

**Is the item there at all?** Ingestion runs after stage-out, on MAAP's
schedule, so a job that just finished may not be indexed for a while.

```python
requests.get(f"{STAC}/collections").json()          # every collection you have
requests.get(f"{STAC}/collections/{BROWSE}/items").json()   # every day in this one
```

**Check the analysis level.** A capped run is structurally identical to a full
one, so nothing on screen distinguishes them:

```python
requests.get(f"{STAC}/collections/{GRANULES}/items/mur-l4-{DAY}-nrt"
             ).json()["properties"].get("mur:analysis_level")
```

`10` means the field is one doubling coarser than production in each
direction. It is smoother, not wrong.

**Can titiler read the file?**

```python
requests.get(f"{TITILER}/collections/{BROWSE}/items/{item_id}/info",
             params={"assets": "sst"}).json()
```

Note that `/info` honours `assets=` but `tilejson.json` does not: it computes
zoom bounds across *all* of an item's assets, so an item mixing a netCDF with
its GeoTIFFs fails to tile for every value of `assets=`. That is why the
granules and the browse rasters are separate items in separate collections.
